In [1]:
from transformers import AutoTokenizer
from datasets import Dataset
import os

In [2]:
datasets = Dataset.load_from_disk("./my_novel_dataset")
tokenizer = AutoTokenizer.from_pretrained("./my_novel_tokenizer")

In [3]:
# 测试分词效果
test_code = datasets['text'][2]
'|'.join([tokenizer.decode(i) for i in tokenizer.encode(test_code)])

'这|可|绝|不是件|容易的事|。|一年|，|二年|，|至少|有|三四|年|；|一|滴汗|，|两|滴汗|，|不知道|多少|万|滴汗|，|才|挣出|那辆车|。|从|风|里|雨|里的|咬牙|，|从|饭|里|茶|里的|自苦|，|才|赚|出|那辆车|。|那辆车|是他的|一切|挣扎|与|困苦|的|总|结果|与|报酬|，|象|身|经|百|战|的|武|士|的一|颗|�|�|�|�|。|在他|赁|人家的车|的时候|，|他从|早|到晚|，|由|东|到|西|，|由|南|到|北|，|象被|人家|抽|着|转|的|�|�|�|�|；|他没有|自己|。|可是|在这种|旋转|之中|，|他的眼|并没有|花|，|心|并没有|乱|，|他老|想|着|远远|的一|辆车|，|可以|使他|自由|，|独|立|，|象|自己的|手|脚的|那么|一辆车|。|有了自己的车|，|他可以|不再|受|拴|车|的人们|的气|，|也|无须|敷衍|别人|；|有|自己的力气|与|洋车|，|睁开眼|就可以|有|饭吃|。|\n|他不|怕|吃|苦|，|也没有|一般|洋车夫的|可以|原谅|而|不便|效|法的|恶|习|，|他的|聪明|和|努力|都|足以|使|他的|志愿|成为|事实|。|假若他|的�|�境|好|一些|，|或|多|受|着点|教|�|�|，|他一|定|不会|落在|"|�|�皮|团|"|⑤|里|，|而且|无论|是|干什么|，|他总|不会|�|�|负|了他的|机会|。|不幸|，|他必须|拉|洋车|；|好|，|在这个|营生|里|他也|证明|出|他的|能力|与|聪明|。|他仿佛|就是在|地狱里|也能|作|个好|鬼|似的|。|生|长|在乡间|，|失去了|父母|与|几|亩|薄|田|，|十|八|岁的时候|便|跑|到城里来|。|带着|乡间|小伙子|的|足|壮|与|诚|实|，|凡是|以|卖力气|就能|吃饭|的事|他几乎|全|作|过了|。|可是|，|不久|他就|看出来|，|拉车|是件|更|容易|挣钱|的事|；|作|别的|苦|工|，|收入|是有|限|的|；|拉车|多|着一些|变|化|与|机会|，|不知道|在什么时候|与|地点|就会|遇到|一些|多|于|所|希望|的报酬|。|自然|，|他也|晓得|这样的|机|遇|不完全|出于|偶|然|，|而|必须|人与车|都得|漂亮|精神|，|有|货|可|卖|才能|遇到|识|货|的人|。|想|了一|想|，|他相信|自己有|

In [4]:
# 添加结束字符：'<|e|>'
tokenizer.add_tokens("<|e|>")
tokenizer.end_ind = tokenizer.encode("<|e|>")[0]
# 测试一下：
tokenizer.decode(tokenizer.encode("<|e|>"))

'<|e|>'

In [5]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader

In [6]:
torch.manual_seed(12046)

# 一些超参数
emb_size = 384          # 模型维度（推荐 256/384/512）
head_size = 8           # 注意力头维度（通常是 emb_size // n_head，建议 64/128）
# n_head = emb_size // head_size # 注意力头数 (自动计算：384/8=48)
n_layer = 6             # Transformer层数（推荐 6/8，13W字不用太深）
sequence_len = 128      # 上下文窗口（推荐 64/128，决定一次看多少字）
learning_rate = 3e-4    # 学习率（小模型用 3e-4 最稳）
eval_iters = 3          # 评估次数（不用改）
batch_size = 100         # 显存足够就上 32/64，不够就 8/4

# 如果有GPU，该脚本将使用GPU进行计算
device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [7]:
# 生成训练标签
def process(data, tokenizer, sequence_len = sequence_len):
    text = data['text']
    # text is list[str]
    inputs, labels = [], []
    for t in text:
        enc = tokenizer.encode(t)
        enc += [tokenizer.end_ind]
        # 有bug，无法处理长度过小的数据
        for i in range(len(enc) - sequence_len):
            inputs.append(enc[i: i + sequence_len])
            labels.append(enc[i + 1: i + 1 + sequence_len])
    return {'inputs': inputs, 'labels': labels}

In [8]:
# 划分数据集
tokenized = datasets.train_test_split(test_size=0.1, seed=1024, shuffle=True)

f = lambda x: process(x, tokenizer)
tokenized = tokenized.map(f, batched=True, remove_columns = datasets.column_names)
tokenized.set_format(type='torch', device = device)

In [9]:
train_loader = DataLoader(tokenized['train'], batch_size = batch_size, shuffle = True)
test_loader = DataLoader(tokenized['test'], batch_size = batch_size, shuffle = True)

In [10]:
@torch.no_grad()
def generate(model, context, tokenizer, max_new_tokens = 300):
    # context: (1, T)
    #out = []
    out = context.tolist()[0]
    model.eval()
    for _ in range(max_new_tokens):
        # 由于注意力机制的长度限制，截断背景
        logits = model(context[:, -sequence_len:])
        probs = F.softmax(logits[:, -1, :], dim=-1)
        # 随机生成文本
        ix = torch.multinomial(probs, num_samples=1)
        # 更新背景
        context = torch.concat((context, ix), dim=-1)
        out.append(ix.item())
        if out[-1] == tokenizer.end_ind:
            break
    model.train()
    return out

In [11]:
def estimate_loss(model):
    re = {}
    # 将模型切换至评估模式
    model.eval()
    re['train'] = _loss(model, train_loader)
    re['test'] = _loss(model, test_loader)
    # 将模型切换至训练模式
    model.train()
    return re

@torch.no_grad()
def _loss(model, data_loader):
    """
    计算模型在不同数据集下面的评估指标
    """
    loss = []
    data_iter= iter(data_loader)
    # 随机使用多个批量数据来预估模型效果
    for k in range(eval_iters):
        data = next(data_iter, None)
        if data is None:
            data_iter = iter(data_loader)
            data = next(data_iter, None)
        inputs, labels = data['inputs'], data['labels']  # (B, T)
        logits = model(inputs)                           # (B, T, vs)
        # 请参考官方文档
        loss.append(F.cross_entropy(logits.transpose(-2, -1), labels).item())
    return torch.tensor(loss).mean().item()

In [12]:
def train_model(model, optimizer, epochs = 10):
    # 记录模型在训练集上的模型损失
    lossi = []
    for epoch in range(epochs):
        for i, data in enumerate(train_loader, 0):
            inputs, labels = data['inputs'], data['labels']  # (B, T)
            optimizer.zero_grad()
            logits = model(inputs)                           # (B, T, vs)
            loss = F.cross_entropy(logits.transpose(-2, -1), labels)
            lossi.append(loss.item())
            loss.backward()
            optimizer.step()
        # 评估模型，并输出结果
        stats = estimate_loss(model)
        train_loss = f'train loss {stats["train"]:.4f}'
        test_loss = f'test loss {stats["test"]:.4f}'
        print(f'epoch {epoch:>2}: {train_loss}, {test_loss}')
    return lossi

In [13]:
def attention(query, key, value, dropout, mask=None):
    # query, key, value: (B, T, H)
    # mask:                 (T, T)
    # output:            (B, T, H)
    B, T, H = query.shape
    scores = query @ key.transpose(-2, -1) / H ** 0.5
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float('-inf'))
    w_att = F.softmax(scores, dim=-1)  # (B, T, T)
    out = w_att @ value                # (B, T, H)
    return out

In [14]:
class MaskedAttention(nn.Module):
    # 单向自注意力

    def __init__(self, emb_size, head_size):
        # emb_size: C, head_size: H
        super().__init__()
        self.key = nn.Linear(emb_size, head_size, bias=False)
        self.query = nn.Linear(emb_size, head_size, bias=False)
        self.value = nn.Linear(emb_size, head_size, bias=False)
        # 定义下三角矩阵
        self.register_buffer('tril', torch.tril(torch.ones(sequence_len, sequence_len)))
        self.dp = nn.Dropout(0.4)

    def forward(self, x):
        # x:   (B, T, C)
        # out: (B, T, H)
        B, T, C = x.shape
        k = self.key(x)    # (B, T, H)
        q = self.query(x)  # (B, T, H)
        v = self.value(x)  # (B, T, H)
        mask = self.tril[:T, :T]
        out = attention(q, k, v, self.dp, mask)
        return out

In [15]:

class MaskedMultiHeadAttention(nn.Module):

    def __init__(self, emb_size, head_size):
        super().__init__()
        # 计算单头注意力的个数
        n_head = emb_size // head_size
        heads = [MaskedAttention(emb_size, head_size) for _ in range(n_head)]
        self.heads = nn.ModuleList(heads)
        self.proj = nn.Linear(emb_size, emb_size)
        self.dp = nn.Dropout(0.4)

    def forward(self, x):
        # x:   (B, T, C)
        # out: (B, T, C)
        out = torch.concat([h(x) for h in self.heads], dim=-1)  # (B, T, C)
        out = self.dp(self.proj(out))                           # (B, T, C)
        return out

In [16]:
class FeedForward(nn.Module):

    def __init__(self, emb_size):
        super().__init__()
        self.ln1 = nn.Linear(emb_size, 4 * emb_size)
        self.ln2 = nn.Linear(4 * emb_size, emb_size)
        self.dp = nn.Dropout(0.4)

    def forward(self, x):
        # x: (B, T, C)
        out = F.gelu(self.ln1(x))     # (B, T, C)
        out = self.dp(self.ln2(out))  # (B, T, C)
        return out

In [17]:
class Block(nn.Module):

    def __init__(self, emb_size, head_size):
        super().__init__()
        self.l1 = nn.LayerNorm(emb_size)
        self.mha = MaskedMultiHeadAttention(emb_size, head_size)
        self.l2 = nn.LayerNorm(emb_size)
        self.ff = FeedForward(emb_size)

    def forward(self, x):
        # x:   (B, T, C)
        # out: (B, T, C)
        # 千万不要使用+=！！！
        x = x + self.mha(self.l1(x))
        x = x + self.ff(self.l2(x))
        return x

In [18]:
class CharGPT(nn.Module):

    def __init__(self, vs):
        super().__init__()
        self.token_emb = nn.Embedding(vs, emb_size)
        self.pos_emb = nn.Embedding(sequence_len, emb_size)
        block = [Block(emb_size, head_size) for _ in range(n_layer)]
        self.blocks = nn.Sequential(*block)
        self.l = nn.LayerNorm(emb_size)
        self.lm = nn.Linear(emb_size, vs)

    def forward(self, x):
        # x: (B, T)
        # logits: (B, T, vs)
        B, T = x.shape
        pos = torch.arange(0, T, dtype=torch.long, device=x.device)
        token_embeddings = self.token_emb(x)        # (B, T, C)
        position_embeddings = self.pos_emb(pos)     # (   T, C)
        h = token_embeddings + position_embeddings  # (B, T, C)
        h = self.blocks(h)                          # (B, T, C)
        logits = self.lm(self.l(h))                 # (B, T, vs)
        return logits

In [19]:
c_model = CharGPT(len(tokenizer)).to(device)
c_model, sum(p.numel() for p in c_model.parameters())

(CharGPT(
   (token_emb): Embedding(8001, 384)
   (pos_emb): Embedding(128, 384)
   (blocks): Sequential(
     (0): Block(
       (l1): LayerNorm((384,), eps=1e-05, elementwise_affine=True)
       (mha): MaskedMultiHeadAttention(
         (heads): ModuleList(
           (0-47): 48 x MaskedAttention(
             (key): Linear(in_features=384, out_features=8, bias=False)
             (query): Linear(in_features=384, out_features=8, bias=False)
             (value): Linear(in_features=384, out_features=8, bias=False)
             (dp): Dropout(p=0.4, inplace=False)
           )
         )
         (proj): Linear(in_features=384, out_features=384, bias=True)
         (dp): Dropout(p=0.4, inplace=False)
       )
       (l2): LayerNorm((384,), eps=1e-05, elementwise_affine=True)
       (ff): FeedForward(
         (ln1): Linear(in_features=384, out_features=1536, bias=True)
         (ln2): Linear(in_features=1536, out_features=384, bias=True)
         (dp): Dropout(p=0.4, inplace=False)
    

In [20]:
context = torch.tensor(tokenizer.encode('祥子'), device = device).unsqueeze(0)
print(''.join(tokenizer.decode(generate(c_model, context, tokenizer))))

祥子可是那的�讲好用手丧厨胡已被�的热着地��快快�人来那几张画缩用~漂亮的车上挂着�小越说正在这个他比爽快驾更加�他并没穷�的茶服软哼祥子不敢再他只好?看了看狱工夫忽然想起还跟着厂子里你堵住��或者弃坐着此外可是祥子嚓付着三天拚枝迷迷严肃可能混晚上搬到前后下的去买人家好容易妈受累的清进两包眼珠铺唱呕特别步儿没有�圣人也只能扛沫�了�州有时她这么�夜里了口气他也许拉起车�祥子把钱刘姑娘老者举出口咕虹高妈的话左官司儿车关票没有人疼痛没打��现在拉车V第他什么也不闪�驼小店她想咧咧谁也不屑于见不上他必须也闹一样营奇怪不及巢那两辆车月的出口先别直入公堂不行满了群安闲高兴花断了在外半死黄村诉诉仔细触并且帮儿车不象个烟袋羞耻他慢慢的他细细看了看锅u灾既然他根本眼中的�个人的膝�快伟大的地方�是怎样的不天桥得更裹递�至多苗啪推门�票不必穷人还不肯堆叹妇女阳的往下落丫头车口上么的味儿两位务这句话义刚一实的得怪不通七八成教虽隔此外祥子摸送汽其余的就是一天并不能的社会脸上个孩子凡是除了几块疲乏凡壶法儿�此编��屈逮顺着讨尊坐在旋转�行劈祥子刚头的上门成家立业�那么大的个子麻利铜烂凑足乱子蓝造她可以��明的喳留着了半天人人春风十多掀木碴儿天天祥子摸五十她把愚一把儿探头这自然他所没在一到事儿注意他似乎跟你医生的要寂寞连自己净了米桃他是争辩�不高兴顶好是只为护了衣裳哪里拐弯挂着些凉微动L�眼看她


In [21]:
estimate_loss(c_model)

{'train': 9.162455558776855, 'test': 9.169906616210938}

In [22]:
l = train_model(c_model, optim.AdamW(c_model.parameters(), lr=learning_rate))

epoch  0: train loss 2.0942, test loss 8.5986
epoch  1: train loss 0.2290, test loss 10.8272
epoch  2: train loss 0.1171, test loss 11.9608
epoch  3: train loss 0.0816, test loss 12.6892
epoch  4: train loss 0.0648, test loss 13.1193
epoch  5: train loss 0.0538, test loss 13.6249
epoch  6: train loss 0.0519, test loss 14.0638
epoch  7: train loss 0.0500, test loss 14.4188
epoch  8: train loss 0.0467, test loss 14.7858
epoch  9: train loss 0.0447, test loss 15.0925


In [23]:
context = torch.tensor(tokenizer.encode('初秋的夜晚，星光叶影里阵阵的小风，祥子抬起头，'), device=device).unsqueeze(0)
print(''.join(tokenizer.decode(generate(c_model, context, tokenizer))))

初秋的夜晚，星光叶影里阵阵的小风，祥子抬起头，看着高远的天河，叹了口气。这么凉爽的天，他的胸脯又是那么宽，可是他觉到空气仿佛不够，胸中非常憋闷。他想坐下痛哭一场。以自己的体格，以自己的忍性，以自己的要强，会让人当作猪狗，会维持不住一个事情，他不只怨恨杨家那一伙人，而渺茫的觉到一种无望，恐怕自己一辈子不会再有什么起色了。拉着铺盖卷，他越走越慢，好象自己已经不是拿起腿就能跑个十里八里的祥子了。
到了大街上，行人已少，可是街灯很亮，他更觉得空旷渺茫，不知道往哪里去好了。上哪儿？自然是回人和厂。心中又有些难过。作买卖的，卖力气的，不怕没有生意，倒怕有了照顾主儿而没作成买卖，象饭铺理发馆进来客人，看了一眼，又走出去那样。祥子明知道上工辞工是常有的事，此处不留爷，自有留爷处。可是，他是低声下气的维持事情，舍着脸为是买上车，而结果还是三天半的事儿，跟那些串惯宅门的老油子一个样，他觉着伤心。他几乎觉得没脸再进人和厂，而给大家当笑话说："瞧瞧瞧，骆驼祥子敢情也是三天半就吹呀，哼！"
不上人和厂，又上哪里去呢？为免得再为这个事思索，他一直走向西安门大街去。人和厂的前脸是三间铺面房，当中的一间作为柜房，只许车夫们进来


In [24]:
torch.save(c_model.state_dict(), "骆驼祥子文本生成.pth")